In [33]:
# Nenhuma instalação extra necessária no Colab
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

In [34]:
df = pd.read_csv("vendas_brutas_loja_moda.csv", encoding="utf-8-sig", dtype=str)

print(f"Shape: {df.shape}")
print(f"\nTipos de dado:\n{df.dtypes}")
print(f"\nNulos por coluna:\n{df.isnull().sum()}")
print(f"\nAmostra:\n")
df.head(10)

Shape: (520, 9)

Tipos de dado:
id_venda           object
data_venda         object
produto            object
quantidade         object
preco_unitario     object
forma_pagamento    object
nome_cliente       object
cpf_cliente        object
vendedor           object
dtype: object

Nulos por coluna:
id_venda             0
data_venda           0
produto              0
quantidade           0
preco_unitario      11
forma_pagamento      0
nome_cliente         0
cpf_cliente          0
vendedor           100
dtype: int64

Amostra:



,id_venda,data_venda,produto,quantidade,preco_unitario,forma_pagamento,nome_cliente,cpf_cliente,vendedor
0,438,"July 18, 2023",Jaqueta Couro M,2,274.91,Cartão de Crédito,Osmar Pinheiro,354.549.480-83,Ana Lima
1,326,"February 19, 2023",calça jeans 42,1,112.84,Cartão de Débito,Eliandro Mesquita,529.912.419-04,Carlos Souza
2,131,29/07/2023,Calça Jeans 42,2,128.8,Débito,Amanda Cavalcante,754.330.365-41,Roberto Dias
3,24,"September 04, 2023",Boné aba curva,4,41.63,Cartão de Débito,Rodrigo Figueiredo,563.421.607-33,Roberto Dias
4,445,20-07-2023,calça jeans 42,1,119.55,Pix,Natalia Brito,615.951.484-65,Carlos Souza
5,251,2023-11-08,vestido floral p,1,81.39,Cartão de Débito,Amanda Cavalcante,754.330.365-41,Fernanda Costa
6,53,18/11/23,Tênis Casual 38,2,147.17,cartão de crédito,Eduardo Pinto,473.829.973-76,Fernanda Costa
7,9,05-09-2023,BLUSA DE FRIO G,1,99.83,Cartao de Credito,DANIEL VIEIRA,808.412.411-82,Ana Lima
8,354,16-06-2023,Sandália rasteira 36,1,68.25,Cartao de Credito,Dilma Couto,379.965.075-27,Carlos Souza
9,309,07-02-2023,vestido floral p,4,88.48,CARTÃO CRÉDITO,Juliana Rocha,413.953.767-24,Ana Lima


In [35]:
linhas_antes = len(df)

df = df.drop_duplicates()

linhas_depois = len(df)
print(f"Duplicatas removidas: {linhas_antes - linhas_depois}")
print(f"Linhas restantes: {linhas_depois}")

Duplicatas removidas: 20
Linhas restantes: 500


In [36]:
from dateutil import parser as dateparser

def parse_data(valor):
    """Tenta interpretar qualquer formato de data e retorna datetime ou NaT"""
    try:
        return dateparser.parse(str(valor), dayfirst=True)
    except Exception:
        return pd.NaT

df["data_venda"] = df["data_venda"].apply(parse_data)
df["data_venda"] = pd.to_datetime(df["data_venda"])

# Remover datas fora do período esperado (2023)
mascara_valida = df["data_venda"].dt.year == 2023
print(f"Datas fora de 2023 removidas: {(~mascara_valida).sum()}")
df = df[mascara_valida].copy()

print(f"\nIntervalo de datas após limpeza:")
print(f"  De: {df['data_venda'].min().date()}")
print(f"  Até: {df['data_venda'].max().date()}")

Datas fora de 2023 removidas: 0

Intervalo de datas após limpeza:
  De: 2023-01-01
  Até: 2023-12-30


In [37]:
import unicodedata

def normalizar_texto(texto):
    """Remove acentos, espaços extras e coloca em title case"""
    if pd.isna(texto):
        return None
    texto = str(texto).strip()
    texto = " ".join(texto.split())  # remove espaços duplos
    # Normaliza acentuação para comparação
    texto_sem_acento = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return texto_sem_acento.title()

# Mapeamento manual dos nomes canônicos (como o produto deve aparecer)
mapa_produtos = {
    "Camiseta Basica M": "Camiseta Básica M",
    "Calca Jeans 42": "Calça Jeans 42",
    "Vestido Floral P": "Vestido Floral P",
    "Tenis Casual 38": "Tênis Casual 38",
    "Blusa De Frio G": "Blusa de Frio G",
    "Short Jeans 40": "Short Jeans 40",
    "Sandalia Rasteira 36": "Sandália Rasteira 36",
    "Jaqueta Couro M": "Jaqueta Couro M",
    "Meia Kit C/3": "Meia Kit C/3",
    "Bone Aba Curva": "Boné Aba Curva",
}

df["produto_normalizado"] = df["produto"].apply(normalizar_texto)
df["produto"] = df["produto_normalizado"].map(mapa_produtos).fillna(df["produto_normalizado"])
df = df.drop(columns=["produto_normalizado"])

print("Produtos após padronização:")
print(df["produto"].value_counts())

Produtos após padronização:
produto
Camiseta Básica M       72
Jaqueta Couro M         61
Blusa de Frio G         58
Sandália Rasteira 36    55
Short Jeans 40          49
Vestido Floral P        44
Calça Jeans 42          43
Boné Aba Curva          37
Tênis Casual 38         34
Meia Kit C/3            30
Meia Kit C/ 3           17
Name: count, dtype: int64


In [38]:
def limpar_preco(valor):
    """Converte string de preço para float, tratando vírgula, nulo, zero e negativo"""
    if pd.isna(valor) or str(valor).strip() in ["", "None", "nan"]:
        return None
    try:
        valor_str = str(valor).replace(",", ".").strip()
        numero = float(valor_str)
        if numero <= 0:
            return None  # zero e negativo são inválidos — serão imputados depois
        return round(numero, 2)
    except ValueError:
        return None

df["preco_unitario"] = df["preco_unitario"].apply(limpar_preco)

# Imputar nulos com a mediana do produto correspondente
mediana_por_produto = df.groupby("produto")["preco_unitario"].median()
df["preco_unitario"] = df.apply(
    lambda row: mediana_por_produto[row["produto"]]
    if pd.isna(row["preco_unitario"]) else row["preco_unitario"],
    axis=1
)

print(f"Nulos restantes em preco_unitario: {df['preco_unitario'].isna().sum()}")
print(f"\nPreço mínimo: R$ {df['preco_unitario'].min():.2f}")
print(f"Preço máximo: R$ {df['preco_unitario'].max():.2f}")

Nulos restantes em preco_unitario: 0

Preço mínimo: R$ 21.38
Preço máximo: R$ 313.99


In [39]:
mapa_pagamento = {
    "cartão de crédito": "Cartão de Crédito",
    "cartao de credito": "Cartão de Crédito",
    "cartão crédito": "Cartão de Crédito",
    "dinheiro": "Dinheiro",
    "pix": "Pix",
    "cartão de débito": "Cartão de Débito",
    "cartao de debito": "Cartão de Débito",
    "débito": "Cartão de Débito",
}

df["forma_pagamento"] = (
    df["forma_pagamento"]
    .str.strip()
    .str.lower()
    .map(mapa_pagamento)
    .fillna("Não informado")
)

print("Formas de pagamento após padronização:")
print(df["forma_pagamento"].value_counts())

Formas de pagamento após padronização:
forma_pagamento
Pix                  139
Cartão de Crédito    135
Cartão de Débito     119
Dinheiro             107
Name: count, dtype: int64


In [40]:
# Nome do cliente: title case e sem espaços extras
df["nome_cliente"] = (
    df["nome_cliente"]
    .str.strip()
    .str.title()
)

# Vendedor: substituir nulos por "Não identificado"
df["vendedor"] = df["vendedor"].fillna("Não identificado")

print("Vendedores após tratamento:")
print(df["vendedor"].value_counts())

Vendedores após tratamento:
vendedor
Fernanda Costa      108
Roberto Dias        102
Ana Lima             97
Não identificado     97
Carlos Souza         96
Name: count, dtype: int64


In [41]:
df["quantidade"] = pd.to_numeric(df["quantidade"], errors="coerce").fillna(1).astype(int)
df["preco_unitario"] = df["preco_unitario"].astype(float)
df["id_venda"] = pd.to_numeric(df["id_venda"], errors="coerce").astype("Int64")

# Coluna de valor total por linha
df["valor_total"] = df["quantidade"] * df["preco_unitario"]

# Colunas de tempo para facilitar análise
df["mes"] = df["data_venda"].dt.month
df["mes_nome"] = df["data_venda"].dt.strftime("%B")
df["dia_semana"] = df["data_venda"].dt.day_name()

print("Tipos finais:")
print(df.dtypes)
print(f"\nValor total de vendas no dataset: R$ {df['valor_total'].sum():,.2f}")

Tipos finais:
id_venda                    Int64
data_venda         datetime64[ns]
produto                    object
quantidade                  int64
preco_unitario            float64
forma_pagamento            object
nome_cliente               object
cpf_cliente                object
vendedor                   object
valor_total               float64
mes                         int32
mes_nome                   object
dia_semana                 object
dtype: object

Valor total de vendas no dataset: R$ 77,927.18


In [42]:
# Exportar CSV limpo (para uso nas próximas etapas)
df.to_csv("vendas_limpas_loja_moda.csv", index=False, encoding="utf-8-sig")

# Exportar XLSX limpo (entrega final ao cliente)
df.to_excel("vendas_limpas_loja_moda.xlsx", index=False)

print("Arquivos salvos:")
print("  vendas_limpas_loja_moda.csv  → para uso interno no projeto")
print("  vendas_limpas_loja_moda.xlsx → entrega ao cliente")
print(f"\nResumo final do dataset limpo:")
print(f"  Linhas: {len(df)}")
print(f"  Colunas: {df.columns.tolist()}")
print(f"  Período: {df['data_venda'].min().date()} a {df['data_venda'].max().date()}")
print(f"  Produtos únicos: {df['produto'].nunique()}")
print(f"  Clientes únicos: {df['nome_cliente'].nunique()}")
print(f"  Nulos restantes:\n\n{df.isnull().sum()}")

Arquivos salvos:
  vendas_limpas_loja_moda.csv  → para uso interno no projeto
  vendas_limpas_loja_moda.xlsx → entrega ao cliente

Resumo final do dataset limpo:
  Linhas: 500
  Colunas: ['id_venda', 'data_venda', 'produto', 'quantidade', 'preco_unitario', 'forma_pagamento', 'nome_cliente', 'cpf_cliente', 'vendedor', 'valor_total', 'mes', 'mes_nome', 'dia_semana']
  Período: 2023-01-01 a 2023-12-30
  Produtos únicos: 11
  Clientes únicos: 81
  Nulos restantes:

id_venda           0
data_venda         0
produto            0
quantidade         0
preco_unitario     0
forma_pagamento    0
nome_cliente       0
cpf_cliente        0
vendedor           0
valor_total        0
mes                0
mes_nome           0
dia_semana         0
dtype: int64
